In [5]:
import pandas as pd

# Convert CSV files into dataframes
beverages_df = pd.read_csv('../data/dept-sample/beverages-pairwise.csv')
produce_df = pd.read_csv('../data/dept-sample/produce-pairwise.csv')
snacks_df = pd.read_csv('../data/dept-sample/snacks-pairwise.csv')
product_df = pd.read_csv('../data/intermediary/product-info.csv')

In [14]:
def compute_hybrid_substitution_score(product_df, pairwise_df, output_csv=None):
    if output_csv:
        with open(output_csv, "w") as f:
            f.write("product_id,substitute_id,score,rank\n")

    pairwise_df_i = pairwise_df.groupby("product_i")
    pairwise_df_j = pairwise_df.groupby("product_j")

    for row in product_df.itertuples(index=False):
        product_id = row.product_id

        product_probs = pairwise_df_i.get_group(product_id) if product_id in pairwise_df_i.groups else pd.DataFrame()
        product_j_probs = pairwise_df_j.get_group(product_id) if product_id in pairwise_df_j.groups else pd.DataFrame()

        # Reverse relationships
        products_rev = product_j_probs.rename(columns={
            'product_i': 'product_j',
            'product_j': 'product_i',
            'P_i': 'P_j',
            'P_j': 'P_i'
        })

        product_probs_df = pd.concat([product_probs, products_rev], ignore_index=True)
        if product_probs_df.empty:
            continue

        # Compute metrics
        product_probs_df['jaccard'] = product_probs_df.apply(lambda x: x.P_ij / (x.P_i + x.P_j - x.P_ij), axis=1)
        product_probs_df['conditional'] = product_probs_df.apply(lambda x: ((x.P_ij / x.P_i) + (x.P_ij / x.P_j)) / 2, axis=1)
        product_probs_df['substitution_index'] = product_probs_df.apply(lambda x: ((x.P_i * x.P_j) - x.P_ij) / (x.P_i * x.P_j), axis=1)

        # Normalize
        for col in ['jaccard', 'conditional', 'substitution_index']:
            product_probs_df[col] = (product_probs_df[col] - product_probs_df[col].min()) / (product_probs_df[col].max() - product_probs_df[col].min())

        # Weighted hybrid score
        product_probs_df['score'] = (
            0.5 * product_probs_df['substitution_index'] +
            0.3 * product_probs_df['jaccard'] +
            0.2 * product_probs_df['conditional']
        )

        # Rank
        product_probs_df['rank'] = product_probs_df['score'].rank(method='dense', ascending=False)
        product_probs_df = product_probs_df.dropna(subset=['score'])
        product_probs_df['rank'] = product_probs_df['score'].rank(method='dense', ascending=False).astype(int)
        # Keep only top 20
        top_substitutes_df = product_probs_df[product_probs_df['rank'] <= 20].copy()

        # Select and rename columns before saving
        top_substitutes_df = top_substitutes_df[['product_i', 'product_j', 'score', 'rank']]
        top_substitutes_df.columns = ['product_id', 'substitute_id', 'score', 'rank']

        # Append to CSV
        if output_csv:
            top_substitutes_df.to_csv(output_csv, mode='a', index=False, header=False)
        else:
            return scored_products_df  # For testing

    print(f"Completed substitute calculations. Saved to {output_csv if output_csv else 'DataFrame'}")

compute_hybrid_substitution_score(product_df, snacks_df, output_csv="../data/dept-sample/obj1/snacks_substitutes.csv")


Completed substitute calculations. Saved to ../data/dept-sample/obj1/snacks_substitutes.csv


In [16]:
# Category validation

import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

def validate_substitutes(subs_df, product_info_df):
    """
    Validate a substitution dataframe by checking if product and substitute belong 
    to the same department and/or aisle.

    Parameters:
    - subs_df: DataFrame with ['product_id', 'substitute_id', 'score', 'rank']
    - product_info_df: DataFrame with ['product_id', 'department', 'aisle']

    Returns:
    - results: dict containing separate and overall validation metrics
    - validated_df: DataFrame with added columns for validation results
    """

    # Map product_id to department and aisle
    dept_map = product_info_df.set_index('product_id')['department'].to_dict()
    aisle_map = product_info_df.set_index('product_id')['aisle'].to_dict()

    # Only evaluate the top 5 substitutes
    subs_df = subs_df[subs_df['rank'] <= 5].copy()
    # Add department and aisle info to subs_df
    subs_df['product_dept'] = subs_df['product_id'].map(dept_map)
    subs_df['substitute_dept'] = subs_df['substitute_id'].map(dept_map)
    subs_df['product_aisle'] = subs_df['product_id'].map(aisle_map)
    subs_df['substitute_aisle'] = subs_df['substitute_id'].map(aisle_map)

    # Check matches
    subs_df['same_department'] = subs_df['product_dept'] == subs_df['substitute_dept']
    subs_df['same_aisle'] = subs_df['product_aisle'] == subs_df['substitute_aisle']
    subs_df['valid_substitution'] = subs_df['same_department'] & subs_df['same_aisle']

    # Compute base true and predicted labels
    y_true = [1] * len(subs_df)  # expecting all to be valid (theoretical ideal)
    dept_pred = subs_df['same_department'].astype(int)
    aisle_pred = subs_df['same_aisle'].astype(int)
    overall_pred = subs_df['valid_substitution'].astype(int)

    # Helper function for metrics
    def compute_metrics(y_pred):
        return {
            'accuracy': y_pred.mean(),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1_score': f1_score(y_true, y_pred, zero_division=0),
            'valid_pairs': y_pred.sum(),
            'total_pairs': len(y_pred)
        }

    # Compute metrics by level
    results = {
        'department_metrics': compute_metrics(dept_pred),
        'aisle_metrics': compute_metrics(aisle_pred),
        'combined_metrics': compute_metrics(overall_pred),
    }

    # Add extra breakdown
    breakdown = subs_df.groupby(['same_department', 'same_aisle']).size().reset_index(name='count')
    results['breakdown'] = breakdown

    return results, subs_df

substitutes_df = pd.read_csv("../data/dept-sample/obj1/snacks_substitutes.csv")

results, validated_df = validate_substitutes(substitutes_df, product_df)

validated_df.to_csv("../data/dept-sample/obj1/snacks_results.csv", index=False)

print("Department-level metrics:")
print(results['department_metrics'])

print("\nAisle-level metrics:")
print(results['aisle_metrics'])

print("\nCombined (Dept + Aisle) metrics:")
print(results['combined_metrics'])

print("\nBreakdown:")
print(results['breakdown'])



Department-level metrics:
{'accuracy': 0.0955596600824304, 'precision': 1.0, 'recall': 0.0955596600824304, 'f1_score': 0.17444903014271346, 'valid_pairs': 24368, 'total_pairs': 255003}

Aisle-level metrics:
{'accuracy': 0.06267377246542198, 'precision': 1.0, 'recall': 0.06267377246542198, 'f1_score': 0.11795486835064671, 'valid_pairs': 15982, 'total_pairs': 255003}

Combined (Dept + Aisle) metrics:
{'accuracy': 0.06267377246542198, 'precision': 1.0, 'recall': 0.06267377246542198, 'f1_score': 0.11795486835064671, 'valid_pairs': 15982, 'total_pairs': 255003}

Breakdown:
   same_department  same_aisle   count
0            False       False  230635
1             True       False    8386
2             True        True   15982
